# Taller: anatomía e ingeniería inversa de archivos ejecutables
**IS753 · Compiladores · Universidad Tecnológica de Pereira**  
**Formatos:** ELF64 (Linux), PE32+ (Windows), Mach-O 64 (macOS) y Java (.class / .jar).  
**Modalidad:** parejas · 2 sesiones de 2 horas + 4 horas de investigación autónoma.  
**Integrantes:** Juan David Cortes, Sebastián Prada, Valentina Lasprilla · **Fecha:** 20/11/2026

## Propósito
Investigar cómo se conserva el resultado de la compilación, cómo se carga y qué información se puede recuperar sin disponer del código fuente. Al terminar, deberá justificar sus conclusiones con bytes, direcciones, instrucciones y pruebas reproducibles.

**Productos:** este cuaderno diligenciado, muestras propias, código reconstruido y evidencias. No se entrega la solución de los retos.

## Objetivos
1. Distinguir formato, arquitectura de CPU, ABI y sistema operativo.
2. Interpretar cabeceras, tablas, secciones, segmentos, símbolos y dependencias.
3. Relacionar desplazamientos en disco con direcciones virtuales.
4. Reconstruir decisiones y cálculos desde ensamblador o bytecode.
5. Contrastar código nativo con bytecode JVM y empaquetado JAR.

Trabaje con programas propios o muestras del docente. La práctica no requiere malware, explotación ni modificar software de terceros. Ejecute únicamente muestras de procedencia conocida; el análisis de otros archivos puede ser estático.

## 1. Plan de trabajo y evidencias
| Etapa | Tiempo | Evidencia |
|---|---:|---|
| Investigación y preparación | 60 min autónomos | Fuentes primarias y entorno |
| Sesión 1: cabeceras y formato nativo | 120 min | Tabla de campos, mapa y direcciones |
| Sesión 2: Java y reconstrucción cruzada | 120 min | Bytecode anotado y función reconstruida |
| Comparación y reporte | 180 min autónomos | Matriz de cuatro formatos y conclusiones |

Cada pareja analiza **en profundidad un formato nativo y Java**. El curso intercambia evidencias para cubrir los otros dos formatos nativos. Identifique quién produjo cada muestra: no presente datos de otra pareja como mediciones propias. No se exige disponer de tres sistemas operativos; una VM o muestras del docente permiten análisis estático.

## 2. Modelo conceptual
| Aspecto | ELF64 | PE32+ | Mach-O 64 | Java Class / JAR |
|---|---|---|---|---|
| Contenido ejecutable | Instrucciones CPU | Instrucciones CPU | Instrucciones CPU | Bytecode JVM en atributos Code |
| Organización | Cabecera, tablas y contenido | DOS, PE/COFF, cabecera opcional y secciones | Cabecera, load commands y segmentos | Class: cabecera, pool, miembros y atributos; JAR: contenedor ZIP |
| Carga | Cabeceras de programa | Cabeceras y secciones | Órdenes de carga | Cargador de clases y JVM |
| Inicio | e_entry | AddressOfEntryPoint, RVA | Habitualmente LC_MAIN | main en la clase elegida; Main-Class en JAR ejecutable |
| Unidad de destino | CPU y ABI | CPU y ABI | CPU y ABI | Versión de class y APIs disponibles |

Una sección agrupa contenido; un segmento ELF/Mach-O describe regiones de carga. PE tiene su propio modelo de secciones y alineaciones. El orden físico puede variar: las tablas y desplazamientos son la autoridad. BSS puede ocupar memoria sin almacenar todos sus ceros en disco. La pila y el heap no son simplemente secciones copiadas del archivo.

Un `.class` no es un ejecutable nativo de 64 bits. La JVM de la máquina puede ser de 64 bits y ejecutar el mismo bytecode, si la versión y las dependencias son compatibles. Un JAR no necesariamente tiene punto de arranque. La JVM puede interpretar y compilar mediante JIT; `java` es a su vez un programa nativo.

## 3. Preparación manual del entorno
El kernel de este cuaderno es **Python 3.10 o posterior**, incluso para las actividades Java. Las celdas usan la biblioteca estándar; no instalan paquetes automáticamente.

| Plataforma | Preparación | Inspección nativa |
|---|---|---|
| Linux | GCC o Clang, binutils; GDB opcional | file, readelf, objdump, nm |
| macOS | Command Line Tools de Xcode; LLDB opcional | file, otool, nm, lipo |
| Windows | MSVC Build Tools y herramientas C++; abrir Jupyter desde Developer Command Prompt | dumpbin; debugger opcional |
| Todas | JDK 17 o posterior, no solamente JRE; incluir bin en PATH | javac, javap, java, jar |

Ghidra es una opción gráfica para el análisis nativo; siga sus requisitos oficiales. No es obligatorio. En Colab se trabaja principalmente con ELF; no se obtendrán Mach-O o PE por cambiar la extensión. En Pythonista se pueden leer bytes de muestras, pero la compilación se realiza en un computador.

Registre versión de compilador, OS y CPU. Compilar en ARM64 y x86-64 produce instrucciones diferentes aunque use el mismo formato.

In [1]:
from pathlib import Path
from dataclasses import dataclass
import platform, shutil, subprocess, struct, hashlib, json, re, zipfile
ROOT = Path('executable_lab').resolve()
ROOT.mkdir(exist_ok=True)
print(platform.platform(), platform.machine())
for tool in ('gcc','clang','cl','readelf','objdump','otool','dumpbin','javac','javap','java','jar'):
    print(f'{tool:10}', shutil.which(tool) or 'No disponible')

def run(args, timeout=30):
    """Argumentos separados, sin shell; muestra la salida y comprueba el resultado."""
    args = [str(x) for x in args]
    if shutil.which(args[0]) is None:
        raise RuntimeError(f'Instale o configure en PATH: {args[0]}')
    p = subprocess.run(args, cwd=ROOT, capture_output=True, text=True,
                       errors='replace', timeout=timeout)
    print(p.stdout, p.stderr)
    if p.returncode:
        raise RuntimeError(f'Comando terminó con código {p.returncode}')
    return p.stdout

Windows-10-10.0.19045-SP0 AMD64
gcc        C:\msys64\mingw64\bin\gcc.EXE
clang      No disponible
cl         No disponible
readelf    C:\msys64\mingw64\bin\readelf.EXE
objdump    C:\msys64\mingw64\bin\objdump.EXE
otool      No disponible
dumpbin    No disponible
javac      C:\Program Files\Common Files\Oracle\Java\javapath\javac.EXE
javap      No disponible
java       C:\Program Files\Common Files\Oracle\Java\javapath\java.EXE
jar        No disponible


In [2]:
ROOT = Path.home() / 'executable_lab'
ROOT.mkdir(exist_ok=True)
print("Carpeta:", ROOT)

Carpeta: C:\Users\Tina_\executable_lab


## 4. Investigación documental
Consulte las referencias del final antes de medir. Para cada formato responda:

1. ¿Cuál es su firma? ¿Qué campos indican arquitectura y versión?
2. ¿Qué tabla describe la carga? ¿Cómo se localiza el código?
3. ¿Dónde se guardan símbolos, importaciones, constantes y depuración?
4. ¿Qué significa el punto de entrada? ¿Por qué puede ser distinto de main?
5. ¿Qué información se pierde al optimizar o retirar símbolos?
6. En Java: ¿qué son constant_pool, descriptor, Code y StackMapTable?
7. ¿Qué requisitos limitan la portabilidad de un JAR?

Registre **URL, sección consultada, afirmación y evidencia propia**. No basta con copiar definiciones.

1. Firma PE = MZ en offset 0 (bytes 4d 5a), verificada en sample_O0.exe, SHA-256 83d357c6af…39705d
PE32+ (64 bits). Campo Machine = 0x8664 (x86-64/AMD64). Número de secciones = 20. Encabezado PE en offset 0x80. Verificado con inspect_header sobre sample_O0.exe.

1. BSS: Size 0x1b80 en memoria, File off 0x0 en disco, sin bandera CONTENTS → reserva memoria sin ocupar archivo.
2. Punto de entrada: 0x13f0 (mainCRTStartup), distinto de main (0x4e5) → hay runtime de arranque antes de main.
3. Símbolos O0: transform, main, bias, banner, workspace conservan nombre por compilar con -g.

**2. ¿Qué tabla describe la carga? ¿Cómo se localiza el código?**
En PE la **Section Table** (en offset 0x188, 20 entradas de 40 bytes) describe cómo
cada sección se carga en memoria (VirtualAddress/RVA y tamaños). El código se localiza
por la sección **.text** (RVA 0x1000) y el punto de entrada (RVA 0x13f0).
Evidencia: objdump -h y objdump -f sobre sample_O0.exe.

**3. ¿Dónde se guardan símbolos, importaciones, constantes y depuración?**
- Símbolos: tabla de símbolos COFF (MinGW los incrusta) → visibles con `nm`: transform, main, bias…
- Importaciones: Import Table / .idata → DLLs KERNEL32.dll y msvcrt.dll (objdump -x, "Import Tables").
- Constantes: sección **.rdata** → ahí vive banner "COMPILADORES_LAB_2026".
- Depuración: secciones **.debug_*** (formato DWARF, por compilar con -g) → .debug_info, .debug_line.
Evidencia: objdump -x y nm sobre sample_O0.exe.

**6. En Java: ¿qué son constant_pool, descriptor, Code y StackMapTable?**
- `constant_pool`: tabla de constantes y referencias simbólicas, con entradas de tamaños
  distintos y tipadas (`Class`, `Methodref`, `NameAndType`, `Utf8`, `String`). Es una red:
  #25 → #8 + #26 → #27 + #28. Su contador vale 68 en nuestra muestra pero hay 67 entradas,
  porque la especificación lo define como "entradas + 1" (el índice 0 no se usa).
- `descriptor`: cadena que codifica tipos. `(I)I` recibe int y devuelve int;
  `([Ljava/lang/String;)V` recibe arreglo de String y no devuelve nada;
  `Ljava/lang/String;` es el tipo String.
- `Code`: atributo de un método con `max_stack`, `max_locals`, el bytecode, la tabla de
  excepciones y atributos anidados. En nuestro `transform`: `stack=2, locals=1`. No todos
  los métodos lo tienen (los abstractos y nativos no).
- `StackMapTable`: marcos que declaran el estado de pila y locales en los puntos donde el
  flujo se reúne, para que el verificador no tenga que inferirlos. En `transform` tiene dos
  entradas, en los offsets 6 y 20, que son exactamente los dos destinos de salto.
Evidencia: `javap -c -p -s -v Sample` sobre `Sample.class`, guardado en `java_evidence.txt`.

**7. ¿Qué requisitos limitan la portabilidad de un JAR?**
Tres, independientes entre sí. La **versión del formato**: el `major` del class debe ser
menor o igual al de la JVM; nuestro major 61 no carga en una JVM 11, que lanzaría
`UnsupportedClassVersionError`. Las **APIs disponibles**: aunque el formato sea aceptado, un
método ausente en esa versión falla en ejecución con `NoSuchMethodError`, así que
compatibilidad de formato no implica compatibilidad de biblioteca. Y las **dependencias
externas**: nuestro pool referencia `StringConcatFactory` vía `invokedynamic`, de modo que
el JAR necesita que esas clases existan en el classpath de destino. Además un JAR solo es
ejecutable si su manifiesto declara `Main-Class`; sin esa línea es una biblioteca.
Evidencia: medición de `--release 11` (major 55) frente a `--release 17` (major 61), y el
`META-INF/MANIFEST.MF` de `sample.jar`.

## 5. Crear una muestra nativa
La primera muestra tiene fuente visible para aprender a relacionar C con el binario. En el reto posterior se intercambian ejecutables sin fuente. `volatile` facilita observar algunos accesos; no garantiza una traducción fija.

In [3]:
C_SOURCE = r"""#include <stdio.h>
#include <stdlib.h>
volatile int bias = 7;
volatile int workspace[1024];
const char banner[] = "COMPILADORES_LAB_2026";
int transform(int x) {
    if (x < 0) return -1;
    if (x % 2 == 0) return x * 3 + bias;
    return x * x - bias;
}
int main(int argc, char **argv) {
    int x = argc > 1 ? atoi(argv[1]) : 4;
    workspace[0] = transform(x);
    printf("%s %d\n", banner, workspace[0]);
    return 0;
}
"""
(ROOT / 'sample.c').write_text(C_SOURCE, encoding='utf-8')
SYSTEM = platform.system()

# Adaptación: usamos gcc (MinGW) en lugar de MSVC, pero seguimos en Windows → produce PE32+.
CC = shutil.which('gcc')
if not CC:
    raise RuntimeError('Se requiere gcc (MinGW)')
run([CC, '-O0', '-g', 'sample.c', '-o', 'sample_O0.exe'])
run([CC, '-O2',       'sample.c', '-o', 'sample_O2.exe'])
NATIVE = ROOT / 'sample_O0.exe'

In [4]:
O0 = ROOT / 'sample_O0.exe'
O2 = ROOT / 'sample_O2.exe'
print("O0:", O0.exists(), "| O2:", O2.exists())

O0: True | O2: True


### Actividad A · inventario y bytes
Ejecute las siguientes celdas solo después de compilar, o asigne NATIVE a una muestra del docente. Registre tamaño, SHA-256 y primeros 64 bytes. La extensión no prueba el formato. Los hashes identifican muestras, no garantizan su seguridad.

In [5]:
def hexdump(data, limit=64):
    for off in range(0, min(len(data), limit), 16):
        part = data[off:off+16]
        chars = ''.join(chr(b) if 32 <= b < 127 else '.' for b in part)
        print(f'{off:08x}  {part.hex(" "):<47}  {chars}')

@dataclass
class Sample:
    path: Path
    def data(self): return self.path.read_bytes()
    def report(self):
        b = self.data()
        return {'archivo': self.path.name, 'bytes': len(b),
                'sha256': hashlib.sha256(b).hexdigest()}

print(Sample(NATIVE).report())
hexdump(NATIVE.read_bytes())

{'archivo': 'sample_O0.exe', 'bytes': 255409, 'sha256': '68774e2b0ab21025732b26d53a69990668ca7d3b034c36670d5ef90318fe5561'}
00000000  4d 5a 90 00 03 00 00 00 04 00 00 00 ff ff 00 00  MZ..............
00000010  b8 00 00 00 00 00 00 00 40 00 00 00 00 00 00 00  ........@.......
00000020  00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00  ................
00000030  00 00 00 00 00 00 00 00 00 00 00 00 80 00 00 00  ................


### Actividad B · detector inicial en Python
Este detector didáctico valida firmas y algunos campos, **no sustituye un parser completo**. Un binario universal Mach-O clásico y Java pueden compartir `CA FE BA BE`: por ello no debe decidirse solo por cuatro bytes. Los archivos `.class` usan big-endian; el Mach-O habitual de equipos actuales usa little-endian. Las muestras ambiguas se reportan como tales.

In [6]:
def inspect_header(path):
    b = Path(path).read_bytes()
    def need(n):
        if len(b) < n: raise ValueError('Archivo truncado')
    need(4)
    if b[:4] == b'\x7fELF':
        need(20)
        if b[4] not in (1,2) or b[5] not in (1,2):
            raise ValueError('Clase u orden ELF no soportado')
        order = '<' if b[5] == 1 else '>'
        return dict(formato='ELF', bits=32 if b[4]==1 else 64,
                    endian='little' if order=='<' else 'big',
                    machine=struct.unpack_from(order+'H', b, 18)[0])
    if b[:2] == b'MZ':
        need(64)
        pe = struct.unpack_from('<I', b, 0x3c)[0]
        need(pe+26)
        if b[pe:pe+4] != b'PE\0\0': raise ValueError('MZ sin firma PE válida')
        machine, sections = struct.unpack_from('<HH', b, pe+4)
        magic = struct.unpack_from('<H', b, pe+24)[0]
        return dict(formato='PE32+' if magic==0x20b else 'PE/otro',
                    machine=machine, sections=sections, pe_offset=pe)
    if b[:4] in (b'\xcf\xfa\xed\xfe', b'\xfe\xed\xfa\xcf'):
        need(32)
        order = '<' if b[:4] == b'\xcf\xfa\xed\xfe' else '>'
        fields = struct.unpack_from(order+'8I', b)
        return dict(formato='Mach-O 64', cpu=fields[1], filetype=fields[3],
                    ncmds=fields[4], sizeofcmds=fields[5])
    if b[:4] == b'\xca\xfe\xba\xbe':
        need(10)
        minor, major, pool_count = struct.unpack_from('>HHH', b, 4)
        return dict(formato='Java Class o Mach-O universal: confirmar con herramienta',
                    minor_candidate=minor, major_candidate=major,
                    constant_pool_count_candidate=pool_count)
    return dict(formato='No reconocido por este detector')

print(inspect_header(NATIVE))

{'formato': 'PE32+', 'machine': 34404, 'sections': 20, 'pe_offset': 128}


In [7]:
b = O0.read_bytes()
pe = struct.unpack_from('<I', b, 0x3c)[0]              # offset del PE (0x80)

print("Firma MZ en 0x00 :", b[0:2].hex(' '))
print("Firma PE en", hex(pe), ":", b[pe:pe+4])

machine_off = pe + 4
machine = struct.unpack_from('<H', b, machine_off)[0]
print("Machine en", hex(machine_off), "bytes", b[machine_off:machine_off+2].hex(' '), "->", hex(machine))

opt = pe + 24                                          # inicio cabecera opcional
opt_size = struct.unpack_from('<H', b, pe+20)[0]
nsec = struct.unpack_from('<H', b, pe+6)[0]

entry_off = opt + 16                                   # AddressOfEntryPoint
assert entry_off + 4 <= len(b)                         # comprobar límites
entry_rva = struct.unpack_from('<I', b, entry_off)[0]
image_base = struct.unpack_from('<Q', b, opt + 24)[0]  # ImageBase (8 bytes, PE32+)
sectab = opt + opt_size                                # inicio de la Section Table

print("EntryPoint en", hex(entry_off), "bytes", b[entry_off:entry_off+4].hex(' '), "-> RVA", hex(entry_rva))
print("ImageBase:", hex(image_base), "| VA real =", hex(image_base + entry_rva))
print("Section Table en", hex(sectab), "con", nsec, "secciones")

Firma MZ en 0x00 : 4d 5a
Firma PE en 0x80 : b'PE\x00\x00'
Machine en 0x84 bytes 64 86 -> 0x8664
EntryPoint en 0xa8 bytes f0 13 00 00 -> RVA 0x13f0
ImageBase: 0x140000000 | VA real = 0x1400013f0
Section Table en 0x188 con 20 secciones


**Por completar:** añada lectura del punto de entrada para el formato nativo escogido. Documente offset, ancho y endian, compruebe límites y contraste con una herramienta. No confunda `LC_MAIN.entryoff` con una dirección virtual ni una RVA PE con un offset del archivo.

| Campo | Offset del campo | Bytes | Interpretación | Herramienta de contraste |
|---|---:|---|---|---|
| Firma | 0x00 | 4d 5a | "MZ" → ejecutable DOS/PE | objdump -f → "file format pei-x86-64" |
| Arquitectura (Machine) | 0x84 | 64 86 | little-endian → 0x8664 = x86-64/AMD64 | objdump -f → "architecture: i386:x86-64" |
| Punto de entrada (AddressOfEntryPoint) | 0xA8 | f0 13 00 00 | RVA 0x13f0; VA = ImageBase 0x140000000 + RVA = 0x1400013f0. **Es una RVA, NO un offset de archivo** | objdump -f → "start address 0x00000001400013f0" |
| Tabla de carga (Section Table) | 0x188 (ver celda) | 20 entradas × 40 bytes | describe las 20 secciones que se cargan en memoria | objdump -h → lista las 20 secciones |

Comparación O0 vs O2 (función transform):

Tamaño binario: casi igual (255.409 vs 254.296 bytes). Las métricas globales apenas cambian porque el 99% del binario es el runtime de MinGW, idéntico en ambos; transform/main son mínimos frente a él.
transform en sí: O0 usa ~24 instrucciones; O2 ~13. O0 guarda x en la pila y lo relee en cada uso; O2 lo mantiene en el registro %ecx sin tocar memoria. O2 elimina el prólogo push %rbp, reemplaza x*3 (tres sumas) por un solo lea, y usa varios ret directos en vez de saltar a un único final.
Inlining: NO ocurrió. transform sigue visible como función independiente en ambos, porque bias es volatile, lo que impide al optimizador fusionar la función.
Símbolos: los nombres se conservan en O0 por -g; el taller advierte que en PE/MSVC la presencia de símbolos puede depender de un PDB externo (aquí usamos MinGW, que sí los incrusta).

## 6. Inspección nativa por sistema
Ejecute únicamente la rama de su plataforma. Conserve la salida completa en archivos de evidencia; seleccione fragmentos y explíquelos en el reporte.

In [8]:
if SYSTEM == 'Linux':
    commands = [['readelf','-h','sample_O0'], ['readelf','-l','sample_O0'],
                ['readelf','-S','sample_O0'], ['readelf','-d','sample_O0'],
                ['nm','-n','sample_O0'], ['objdump','-d','sample_O0']]
elif SYSTEM == 'Darwin':
    commands = [['otool','-hv','sample_O0'], ['otool','-l','sample_O0'],
                ['otool','-L','sample_O0'], ['nm','-n','sample_O0'],
                ['otool','-tvV','sample_O0']]
elif SYSTEM == 'Windows':
    commands = [['objdump','-f','sample_O0.exe'],   # cabecera general del archivo
                ['objdump','-h','sample_O0.exe'],   # tabla de secciones (el mapa)
                ['objdump','-x','sample_O0.exe'],   # todo: cabeceras, secciones, símbolos
                ['nm','sample_O0.exe'],             # tabla de símbolos (nombres)
                ['objdump','-d','sample_O0.exe']]   # desensamblado (código en ASM)
else:
    commands = []
for i, cmd in enumerate(commands):
    output = run(cmd)
    (ROOT / f'native_evidence_{i}.txt').write_text(output, encoding='utf-8')


sample_O0.exe:     file format pei-x86-64
architecture: i386:x86-64, flags 0x0000013b:
HAS_RELOC, EXEC_P, HAS_DEBUG, HAS_SYMS, HAS_LOCALS, D_PAGED
start address 0x00000001400013f0

 

sample_O0.exe:     file format pei-x86-64

Sections:
Idx Name          Size      VMA               LMA               File off  Algn
  0 .text         00007148  0000000140001000  0000000140001000  00000600  2**4
                  CONTENTS, ALLOC, LOAD, READONLY, CODE, DATA
  1 .data         000000e0  0000000140009000  0000000140009000  00007800  2**4
                  CONTENTS, ALLOC, LOAD, DATA
  2 .rdata        00001020  000000014000a000  000000014000a000  00007a00  2**4
                  CONTENTS, ALLOC, LOAD, READONLY, DATA
  3 .pdata        00000474  000000014000c000  000000014000c000  00008c00  2**2
                  CONTENTS, ALLOC, LOAD, READONLY, DATA
  4 .xdata        00000434  000000014000d000  000000014000d000  00009200  2**2
                  CONTENTS, ALLOC, LOAD, READONLY, DATA
  5 .bss    

### Actividad C · mapa del archivo y de memoria
1. Ubique instrucciones de `transform`, `banner`, `bias` y `workspace`; si no se conservan nombres, use referencias y el binario sin optimizar.
2. Dibuje bloques con offsets y tamaños **medidos**, y otro mapa con direcciones virtuales y permisos. Puede usar una tabla o un dibujo propio.
3. Localice una dependencia externa y una llamada que la use.
4. Explique el tamaño en archivo y en memoria de BSS. No asuma que cada símbolo tiene una sección exclusiva.
5. Compare O0/O2: tamaño, llamadas, saltos, símbolos e inlining. La presencia de símbolos en PE puede depender de un PDB externo.

| Región | Offset archivo | Tamaño archivo | Dirección virtual | Tamaño memoria | Permisos | Evidencia |
|---|---:|---:|---:|---:|---|---|
| Código (.text) | 0x600 | 0x7148 | 0x140001000 | 0x7148 | R-X (lee/ejecuta) | objdump -h; nm: transform @ 0x1400014a4 |
| Constantes (.rdata) | 0x7a00 | 0x1020 | 0x14000a000 | 0x1020 | R-- (solo lectura) | objdump -h; banner @ 0x14000a000 |
| Datos (.data) | 0x7800 | 0xe0 | 0x140009000 | 0xe0 | RW- (lee/escribe) | objdump -h; bias @ 0x140009000 (__data_start__) |
| Ceros iniciales (.bss) | — sin CONTENTS | 0 | 0x14000e000 | 0x1b80 | RW- | objdump -h: ALLOC sin CONTENTS; workspace @ 0x14000e040 |

### Actividad D · convertir una dirección
Seleccione un byte de instrucciones en una región respaldada por archivo:

- ELF: `offset = p_offset + (VA - p_vaddr)` dentro de un PT_LOAD y de su intervalo respaldado por p_filesz.
- Mach-O: `offset = fileoff + (VA - vmaddr)` dentro del segmento y su filesize.
- PE: `offset = PointerToRawData + (RVA - VirtualAddress)` dentro de la sección y su SizeOfRawData; `VA = base_real + RVA`.

Las direcciones del análisis estático pueden diferir de las reales por ASLR/PIE. Muestre un cálculo con datos de su muestra y compruebe los bytes. ¿Por qué no funciona la conversión para todo BSS?

In [9]:
image_base = 0x140000000
# Región elegida: sección .text (código), respaldada por archivo
region_va     = 0x140001000     # VMA de .text              (objdump -h)
region_offset = 0x600           # File off / PointerToRawData (objdump -h)
region_filesz = 0x7148          # SizeOfRawData de .text     (objdump -h)

# Byte objetivo: primera instrucción de transform
target_va = 0x1400014a4         # VA de transform           (nm / objdump -d)

delta = target_va - region_va
assert 0 <= delta < region_filesz          # dentro de la sección y respaldado por archivo
offset = region_offset + delta
print("offset de archivo:", hex(offset))
print("bytes:", O0.read_bytes()[offset:offset+16].hex(' '))

offset de archivo: 0xaa4
bytes: 55 48 89 e5 89 4d 10 83 7d 10 00 79 07 b8 ff ff


La conversión no sirve para el .bss porque esa sección no tiene datos en el archivo
(ALLOC sin CONTENTS, SizeOfRawData = 0): sus bytes no existen en disco, solo se
reservan en memoria al cargar. No hay offset de archivo al cual mapear la dirección.

In [10]:
import re, subprocess

def analizar_silencioso(exe):
    # Corremos objdump SIN el print ruidoso de run()
    p = subprocess.run(['objdump', '-d', str(exe)], cwd=ROOT,
                       capture_output=True, text=True, errors='replace')
    disasm = p.stdout
    return {
        'tamaño_bytes': exe.stat().st_size,
        'transform_visible': '<transform>:' in disasm,
        'main_visible': '<main>:' in disasm,
        'num_call': len(re.findall(r'\bcall\b', disasm)),
        'num_jmp_jcc': len(re.findall(r'\bj[a-z]+\b', disasm)),
    }

o0 = analizar_silencioso(O0)
o2 = analizar_silencioso(O2)

print(f"{'métrica':20} {'O0':>12} {'O2':>12}")
print("-" * 46)
for k in o0:
    print(f"{k:20} {str(o0[k]):>12} {str(o2[k]):>12}")

métrica                        O0           O2
----------------------------------------------
tamaño_bytes               255409       254296
transform_visible            True         True
main_visible                 True         True
num_call                      333          333
num_jmp_jcc                  1255         1252


In [11]:
import subprocess, re

def extraer_transform(exe):
    p = subprocess.run(['objdump', '-d', str(exe)], cwd=ROOT,
                       capture_output=True, text=True, errors='replace')
    lineas = p.stdout.splitlines()
    out, dentro = [], False
    for ln in lineas:
        if '<transform>:' in ln:
            dentro = True
        elif dentro and re.match(r'^[0-9a-f]+ <', ln):  # empezó otra función
            break
        if dentro:
            out.append(ln)
    return out

print("===================  transform en O0  ===================")
print("\n".join(extraer_transform(O0)))
print("\n===================  transform en O2  ===================")
print("\n".join(extraer_transform(O2)))

===================  transform en O0  ===================
00000001400014a4 <transform>:
   1400014a4:	55                   	push   %rbp
   1400014a5:	48 89 e5             	mov    %rsp,%rbp
   1400014a8:	89 4d 10             	mov    %ecx,0x10(%rbp)
   1400014ab:	83 7d 10 00          	cmpl   $0x0,0x10(%rbp)
   1400014af:	79 07                	jns    1400014b8 <transform+0x14>
   1400014b1:	b8 ff ff ff ff       	mov    $0xffffffff,%eax
   1400014b6:	eb 2b                	jmp    1400014e3 <transform+0x3f>
   1400014b8:	8b 45 10             	mov    0x10(%rbp),%eax
   1400014bb:	83 e0 01             	and    $0x1,%eax
   1400014be:	85 c0                	test   %eax,%eax
   1400014c0:	75 13                	jne    1400014d5 <transform+0x31>
   1400014c2:	8b 55 10             	mov    0x10(%rbp),%edx
   1400014c5:	89 d0                	mov    %edx,%eax
   1400014c7:	01 c0                	add    %eax,%eax
   1400014c9:	01 c2                	add    %eax,%edx
   1400014cb:	8b 05 2f 7b 00 00    	mov 

In [12]:
JDK_BIN = Path(r"C:\Program Files\Java\jdk-17\bin")

# Rutas completas a cada herramienta Java que usaremos
JAVAC = JDK_BIN / "javac.exe"
JAVAP = JDK_BIN / "javap.exe"
JAVA  = JDK_BIN / "java.exe"
JAR   = JDK_BIN / "jar.exe"

# Verificación
for nombre, ruta in [("javac", JAVAC), ("javap", JAVAP), ("java", JAVA), ("jar", JAR)]:
    print(f"{nombre:7}", "OK" if ruta.exists() else "FALTA", "→", ruta)

# Confirmamos la versión de ESTE javac (no el del PATH)
print("\nVersión de este JDK:")
print(run([str(JAVAC), "-version"]))

javac   OK → C:\Program Files\Java\jdk-17\bin\javac.exe
javap   OK → C:\Program Files\Java\jdk-17\bin\javap.exe
java    OK → C:\Program Files\Java\jdk-17\bin\java.exe
jar     OK → C:\Program Files\Java\jdk-17\bin\jar.exe

Versión de este JDK:
javac 17.0.6
 
javac 17.0.6



## 7. Java: formato Class, bytecode y JAR
Esquema lógico de un Class, en orden: **magic → minor/major → constant_pool_count y constant_pool → access_flags, this_class, super_class → interfaces → fields → methods → attributes**. Hay contadores y longitudes entre estructuras. El pool tiene entradas de tamaños distintos; `long` y `double` ocupan dos índices. No puede saltarse multiplicando el contador por un tamaño fijo.

Un método puede tener un atributo `Code` con `max_stack`, `max_locals`, bytecode, excepciones y atributos anidados. No todos los métodos tienen Code: los abstractos y nativos son ejemplos. Los descriptores expresan tipos: investigue `(I)I`, `([Ljava/lang/String;)V` y `Ljava/lang/String;`.

El JAR agrupa clases y recursos usando ZIP. El manifiesto puede declarar `Main-Class`; no transforma el bytecode en código nativo.

In [13]:
# Adaptación: el PATH solo expone java/javac (javapath de Oracle, sin javap ni jar).
# Invocamos el JDK 17 por ruta completa.
JDK_BIN = Path(r"C:\Program Files\Java\jdk-17\bin")
JAVAC = JDK_BIN / "javac.exe"
JAVAP = JDK_BIN / "javap.exe"
JAVA  = JDK_BIN / "java.exe"
JAR   = JDK_BIN / "jar.exe"

JAVA_SOURCE = r"""public class Sample {
    static int bias = 7;
    static int transform(int x) {
        if (x < 0) return -1;
        if (x % 2 == 0) return x * 3 + bias;
        return x * x - bias;
    }
    public static void main(String[] args) {
        int x = args.length > 0 ? Integer.parseInt(args[0]) : 4;
        System.out.println("COMPILADORES_LAB_2026 " + transform(x));
    }
}
"""
(ROOT / 'Sample.java').write_text(JAVA_SOURCE, encoding='utf-8')
run([str(JAVAC), '-g', 'Sample.java'])
java_listing = run([str(JAVAP), '-classpath', str(ROOT), '-c', '-p', '-s', '-v', 'Sample'])
(ROOT / 'java_evidence.txt').write_text(java_listing, encoding='utf-8')
hexdump((ROOT / 'Sample.class').read_bytes())
print(inspect_header(ROOT / 'Sample.class'))

 
Classfile /C:/Users/Tina_/executable_lab/Sample.class
  Last modified 20/11/2026; size 1307 bytes
  SHA-256 checksum 67d288243daa39d60aa28fdecf171b5b57f5999e22191b6a2e11dd78934431bc
  Compiled from "Sample.java"
public class Sample
  minor version: 0
  major version: 61
  flags: (0x0021) ACC_PUBLIC, ACC_SUPER
  this_class: #8                          // Sample
  super_class: #2                         // java/lang/Object
  interfaces: 0, fields: 1, methods: 4, attributes: 3
Constant pool:
   #1 = Methodref          #2.#3          // java/lang/Object."<init>":()V
   #2 = Class              #4             // java/lang/Object
   #3 = NameAndType        #5:#6          // "<init>":()V
   #4 = Utf8               java/lang/Object
   #5 = Utf8               <init>
   #6 = Utf8               ()V
   #7 = Fieldref           #8.#9          // Sample.bias:I
   #8 = Class              #10            // Sample
   #9 = NameAndType        #11:#12        // bias:I
  #10 = Utf8               Sample
  #

### Actividad E · desensamblar Java
1. Lea minor/major y compruebe la versión en la especificación, sin asumir que el número coincide con la versión comercial de Java.
2. Identifique cinco entradas del pool: clase, método, nombre/tipo y texto. Muestre índices y referencias.
3. Extraiga `transform` y anote operaciones sobre pila, variables locales y saltos.
4. Simule el recorrido para una entrada negativa, una par y una impar. En cada instrucción registre la pila **antes y después**.
5. Dibuje el grafo de flujo de control con bloques básicos. ¿Qué comparación C representa cada salto? Un salto puede expresar la condición complementaria del código fuente.
6. Distinga el `main` de la inicialización estática `<clinit>`, si aparece.

| Offset bytecode | Instrucción | Pila antes | Pila después | Próximo offset y motivo |
|---:|---|---|---|---|
| | | | | |

**Experimento:** compile sin depuración en una carpeta separada y compare con la clase original. ¿Qué metadatos desaparecen y cuáles permanecen necesarios para ejecutar?

**1. Versión del formato**
minor version = 0, major version = 61 (bytes `00 00 00 3d`, big-endian, offsets 4-7).
major 61 corresponde a **Java SE 17** (la numeración empieza en 45 = Java 1.1; 61 - 44 = 17).
El número del formato NO coincide con la versión comercial.
Contraste: compilado con javac 17.0.6 invocado por ruta completa.
Nota: constant_pool_count = 68 (bytes `00 44`) pero el pool termina en #67, porque la
especificación define el contador como "entradas + 1" (el índice 0 no se usa).

**2. Cinco entradas del pool y sus referencias**
| Índice | Tipo | Apunta a | Significado |
|---|---|---|---|
| #8 | Class | #10 | la clase `Sample` |
| #25 | Methodref | #8.#26 | el método `Sample.transform:(I)I` |
| #26 | NameAndType | #27:#28 | nombre `transform` + descriptor `(I)I` |
| #27 | Utf8 | — | el texto literal `transform` |
| #60 | String | #61 | la cadena `COMPILADORES_LAB_2026\u0001` |

Observación: el pool es una red de referencias, no una lista plana. Para resolver una
sola llamada (#25) la JVM recorre #25 → #8 (clase) → #10 (texto "Sample"), y
#25 → #26 → #27 (nombre) + #28 (descriptor "(I)I"). Ningún texto se duplica.
Descriptores observados: `(I)I` = recibe int y devuelve int; `([Ljava/lang/String;)V`
= recibe arreglo de String y no devuelve nada; `Ljava/lang/String;` = tipo String.

**6. main frente a <clinit>**
Aparecen cuatro métodos: `Sample()` (constructor), `transform(int)`, `main(String[])`
y `static {}` = **<clinit>**.
- `main` es el punto de arranque que invoca el lanzador `java`; es público y estático.
- `<clinit>` NO lo llama nadie del programa: lo ejecuta el cargador de clases
  automáticamente, antes del primer uso de la clase. Su único trabajo aquí es
  `bipush 7; putstatic bias` → asignar `bias = 7`.
Contraste con C: allí `bias = 7` era un **dato** en la sección `.data` del archivo;
en Java es **código** que corre al cargar la clase. El mismo valor del fuente acaba
en categorías distintas según el formato.

**3. transform: pila, variables locales y saltos**

Cabecera del Code: `stack=2, locals=1, args_size=1`.
- `locals=1`: una sola variable local, el parámetro `x` en el slot 0
  (LocalVariableTable: Slot 0, Name x, Signature I, válido de 0 a 28).
- `stack=2`: la pila de operandos nunca pasa de 2 valores a la vez.
- `args_size=1`: recibe un argumento. No hay `this` porque es ACC_STATIC.

Operaciones sobre la pila: `iload_0` empuja x; `iconst_2`/`iconst_3`/`iconst_m1`
empujan constantes; `irem`, `imul`, `iadd`, `isub` consumen dos valores y dejan uno;
`getstatic #7` empuja el valor del campo `bias`; `ireturn` consume uno y retorna.

Saltos (los dos son condicionales, no hay `goto` en este método):
- offset 1: `ifge 6` → bifurca según el signo de x.
- offset 9: `ifne 20` → bifurca según la paridad de x.

El `StackMapTable` tiene 2 entradas, y sus offsets absolutos son **6 y 20**:
exactamente los dos destinos de salto. Son los marcos que el verificador necesita
en los puntos donde el flujo se reúne.

**4. Simulación de la pila (bias = 7)**

Caso x = -3 (negativa):

| Offset | Instrucción | Pila antes | Pila después | Próximo offset y motivo |
|---:|---|---|---|---|
| 0 | iload_0 | [] | [-3] | 1, secuencial |
| 1 | ifge 6 | [-3] | [] | 4, porque -3 >= 0 es falso → no salta |
| 4 | iconst_m1 | [] | [-1] | 5, secuencial |
| 5 | ireturn | [-1] | [] | retorna -1 |

Caso x = 4 (par):

| Offset | Instrucción | Pila antes | Pila después | Próximo offset y motivo |
|---:|---|---|---|---|
| 0 | iload_0 | [] | [4] | 1, secuencial |
| 1 | ifge 6 | [4] | [] | 6, porque 4 >= 0 es cierto → salta |
| 6 | iload_0 | [] | [4] | 7, secuencial |
| 7 | iconst_2 | [4] | [4, 2] | 8, secuencial |
| 8 | irem | [4, 2] | [0] | 9, deja 4 % 2 = 0 |
| 9 | ifne 20 | [0] | [] | 12, porque 0 != 0 es falso → no salta |
| 12 | iload_0 | [] | [4] | 13, secuencial |
| 13 | iconst_3 | [4] | [4, 3] | 14, secuencial |
| 14 | imul | [4, 3] | [12] | 15, deja 4 * 3 = 12 |
| 15 | getstatic #7 | [12] | [12, 7] | 18, empuja bias = 7 |
| 18 | iadd | [12, 7] | [19] | 19, deja 12 + 7 = 19 |
| 19 | ireturn | [19] | [] | retorna 19 |

Caso x = 5 (impar):

| Offset | Instrucción | Pila antes | Pila después | Próximo offset y motivo |
|---:|---|---|---|---|
| 0 | iload_0 | [] | [5] | 1, secuencial |
| 1 | ifge 6 | [5] | [] | 6, porque 5 >= 0 es cierto → salta |
| 6 | iload_0 | [] | [5] | 7, secuencial |
| 7 | iconst_2 | [5] | [5, 2] | 8, secuencial |
| 8 | irem | [5, 2] | [1] | 9, deja 5 % 2 = 1 |
| 9 | ifne 20 | [1] | [] | 20, porque 1 != 0 es cierto → salta |
| 20 | iload_0 | [] | [5] | 21, secuencial |
| 21 | iload_0 | [5] | [5, 5] | 22, secuencial |
| 22 | imul | [5, 5] | [25] | 23, deja 5 * 5 = 25 |
| 23 | getstatic #7 | [25] | [25, 7] | 26, empuja bias = 7 |
| 26 | isub | [25, 7] | [18] | 27, deja 25 - 7 = 18 |
| 27 | ireturn | [18] | [] | retorna 18 |

La profundidad máxima observada es 2 en los tres recorridos, lo que confirma el
`stack=2` declarado en la cabecera del Code.

**5. Grafo de flujo de control y correspondencia con el C**

Bloques básicos:

- **B0** (offsets 0-1): `iload_0`, `ifge 6`. Termina en salto condicional.
- **B1** (offsets 4-5): `iconst_m1`, `ireturn`. Retorna -1.
- **B2** (offsets 6-9): `iload_0`, `iconst_2`, `irem`, `ifne 20`.
- **B3** (offsets 12-19): calcula `x * 3 + bias` y retorna.
- **B4** (offsets 20-27): calcula `x * x - bias` y retorna.

Aristas:

        B0
       /   \
   (x<0)   (x>=0)
     B1      B2
            /   \
       (par)   (impar)
         B3      B4

- B0 → B1 cuando NO salta (x < 0); B0 → B2 cuando salta (x >= 0).
- B2 → B3 cuando NO salta (x % 2 == 0); B2 → B4 cuando salta (x % 2 != 0).

**Cada salto expresa la condición COMPLEMENTARIA del fuente.** El C dice
`if (x < 0) return -1;`, pero el bytecode emite `ifge` (salta si es mayor o igual):
salta cuando la condición del C es *falsa*, para esquivar el cuerpo del `if`. Igual
con `if (x % 2 == 0)`, que se compila como `ifne` (salta si es distinto de cero).
Es decir, leer `ifge` como "si x >= 0" invierte el sentido del código original: el
salto no implementa la condición, implementa su negación.

In [14]:
# Mismo arreglo: JDK 17 por ruta completa (el PATH no tiene javap ni jar).
JDK_BIN = Path(r"C:\Program Files\Java\jdk-17\bin")
JAVAC, JAVAP, JAVA, JAR = (JDK_BIN/"javac.exe", JDK_BIN/"javap.exe",
                           JDK_BIN/"java.exe", JDK_BIN/"jar.exe")

no_debug = ROOT / 'java_no_debug'
no_debug.mkdir(exist_ok=True)
run([str(JAVAC), '-g:none', '-d', str(no_debug), 'Sample.java'])
run([str(JAVAP), '-classpath', str(no_debug), '-c', '-p', '-v', 'Sample'])
run([str(JAR), '--create', '--file', 'sample.jar', '--main-class', 'Sample', 'Sample.class'])
with zipfile.ZipFile(ROOT / 'sample.jar') as z:
    print(z.namelist())
    print(z.read('META-INF/MANIFEST.MF').decode('utf-8'))
run([str(JAVA), '-jar', 'sample.jar', '4'])

 
Classfile /C:/Users/Tina_/executable_lab/java_no_debug/Sample.class
  Last modified 20/11/2026; size 1054 bytes
  SHA-256 checksum 8a2e6a2f0982b0a786281f4f24a6f82612e60f89df10ee82fcc4f4c052bdff4e
public class Sample
  minor version: 0
  major version: 61
  flags: (0x0021) ACC_PUBLIC, ACC_SUPER
  this_class: #8                          // Sample
  super_class: #2                         // java/lang/Object
  interfaces: 0, fields: 1, methods: 4, attributes: 2
Constant pool:
   #1 = Methodref          #2.#3          // java/lang/Object."<init>":()V
   #2 = Class              #4             // java/lang/Object
   #3 = NameAndType        #5:#6          // "<init>":()V
   #4 = Utf8               java/lang/Object
   #5 = Utf8               <init>
   #6 = Utf8               ()V
   #7 = Fieldref           #8.#9          // Sample.bias:I
   #8 = Class              #10            // Sample
   #9 = NameAndType        #11:#12        // bias:I
  #10 = Utf8               Sample
  #11 = Utf8       

'COMPILADORES_LAB_2026 19\n'

**Resultado del experimento (-g vs -g:none)**

| | Con `-g` | Con `-g:none` |
|---|---|---|
| Tamaño del .class | 1.307 bytes | 1.054 bytes |
| Entradas del constant pool | 67 | 58 |
| Atributos de clase | 3 | 2 |

**Qué desaparece:** `LineNumberTable` (mapeaba offsets de bytecode a líneas del fuente),
`LocalVariableTable` (guardaba los nombres `x`, `args`, `this` y sus firmas) y el atributo
`SourceFile` (el nombre `Sample.java`). Con ellos se van las entradas del pool que solo
existían para nombrarlos. `javap` ya no imprime la línea `Compiled from "Sample.java"`.

**Qué permanece y por qué es necesario para ejecutar:**
- Los **nombres de métodos y campos** y sus **descriptores** (`transform`, `main`, `bias`,
  `(I)I`). No son información de depuración: viven en el constant pool y la JVM los necesita
  para resolver `invokestatic #25` en tiempo de ejecución.
- El `StackMapTable`, porque lo exige el verificador de bytecode, no el depurador.
- El atributo `Code` de cada método, y `BootstrapMethods` (lo requiere el `invokedynamic`
  de la concatenación de cadenas).

**Observación clave:** el bytecode de `transform` es **idéntico** en las dos versiones,
offset por offset (0 a 27). `-g:none` no cambia el programa, solo le quita metadatos.
Por eso un `.class` sin depuración sigue siendo perfectamente legible con `javap`.

### Actividad F · probar compatibilidad
Investigue `javac --release`. Si su JDK lo permite, genere otra clase con `--release 17` en una carpeta distinta, mida major y compare. No necesita instalar una JVM antigua: argumente qué pasaría al cargar una versión no admitida. ¿La compatibilidad del formato garantiza que todas las APIs y bibliotecas requeridas existan?

Explique por qué cambiar `.jar` por `.exe` o `.class` por `.o` no convierte formatos. Como ampliación documental, investigue compilación AOT de Java: ¿qué cambiaría en la portabilidad y en el formato de salida?

In [15]:
def version_de(class_path):
    b = Path(class_path).read_bytes()
    minor, major = struct.unpack_from('>HH', b, 4)   # big-endian, offsets 4-7
    return f"minor={minor}, major={major}"

for rel in ('17', '11'):
    d = ROOT / f'java_release_{rel}'
    d.mkdir(exist_ok=True)
    run([str(JAVAC), '--release', rel, '-d', str(d), 'Sample.java'])

print("Clase original :", version_de(ROOT / 'Sample.class'))
print("--release 17   :", version_de(ROOT / 'java_release_17' / 'Sample.class'))
print("--release 11   :", version_de(ROOT / 'java_release_11' / 'Sample.class'))

 
 
Clase original : minor=0, major=61
--release 17   : minor=0, major=61
--release 11   : minor=0, major=55


**Medición de `javac --release`**

| Compilación | minor | major | Versión Java |
|---|---|---|---|
| `javac -g` (por defecto, JDK 17) | 0 | 61 | Java SE 17 |
| `javac --release 17` | 0 | 61 | Java SE 17 |
| `javac --release 11` | 0 | 55 | Java SE 11 |

`--release N` hace dos cosas a la vez: fija el `major` del archivo class y restringe la
API visible a la de esa versión. Con `--release 17` el resultado es idéntico al nuestro
porque nuestro JDK *es* el 17; con `--release 11` el major baja a 55, lo que confirma el
desfase de 44 respecto a la versión comercial.

**¿Qué pasaría al cargar una versión no admitida?** La JVM lanza
`UnsupportedClassVersionError` antes de ejecutar nada. Una JVM admite archivos con major
menor o igual al suyo: una JVM 11 no puede cargar nuestro major 61, pero una JVM 17 sí
carga el major 55. La compatibilidad es hacia atrás, no hacia adelante.

**¿La compatibilidad del formato garantiza que existan las APIs?** No. Son dos cosas
distintas. Un `.class` puede tener un major que la JVM acepta y fallar en ejecución con
`NoSuchMethodError` o `NoClassDefFoundError` si invoca un método que esa versión no tiene,
o si falta una biblioteca del classpath. `--release` protege de esto en compilación
precisamente porque también limita la API; usar solo `-target` no lo haría.

**¿Por qué renombrar extensiones no convierte formatos?** Porque el formato está en los
bytes, no en el nombre. Nuestras propias mediciones lo muestran: `sample_O0.exe` empieza
con `4d 5a` (MZ) y tiene su encabezado PE en 0x80; `Sample.class` empieza con
`ca fe ba be` y es big-endian; `sample.jar` es un ZIP. Si renombramos el JAR a `.exe`,
Windows busca la firma MZ, no la encuentra y se niega a cargarlo. Y un `.o` es un objeto
reubicable, sin punto de entrada ni secciones mapeadas: ni siquiera es ejecutable.
La extensión es una convención para el escritorio; el cargador lee la firma.

**Ampliación: compilación AOT.** Un compilador anticipado (por ejemplo `native-image` de
GraalVM) traduce el bytecode a código máquina y produce un **ejecutable nativo**: en
Windows sería un PE32+ como nuestro `sample_O0.exe`, con sus secciones y su punto de
entrada. Se gana arranque instantáneo y no hace falta JVM instalada. Se pierde justo lo
que hace portable a Java: el artefacto deja de ser un `.class` independiente de plataforma
y pasa a necesitar una compilación por cada sistema y arquitectura. También se pierde
flexibilidad en tiempo de ejecución, porque la reflexión y la carga dinámica de clases
deben declararse por adelantado.

## 8. Reto de ingeniería inversa entre parejas
**Preparación privada (no incluir la fuente en el material que se intercambia):** cada pareja escribe una función determinista `int mystery(int x)` con una condición y un bucle acotado. Use entradas de 0 a 20 y resultados pequeños para evitar diferencias de overflow entre C y Java. Genere una muestra nativa y una Java equivalentes. El programa debe aceptar un entero por argumento y mostrar el resultado.

1. Entregue a otra pareja los binarios, plataforma, arquitectura, rango válido y SHA-256; conserve fuente y resultados de referencia.
2. La pareja receptora comienza con análisis estático, formula una hipótesis y reconstruye la función en Python o pseudocódigo preciso.
3. Anote saltos, límites del bucle, operaciones y llamadas. Separe observaciones de inferencias.
4. Compare predicciones contra la muestra autorizada usando al menos ocho entradas, incluidas fronteras. Guarde resultados esperados y observados.
5. Revele la fuente al final. Explique qué recuperó y qué no: equivalencia de comportamiento no implica recuperar texto, nombres ni comentarios originales.

**Roles:** A analiza primero el nativo y B Java; luego intercambian conclusiones. No se evalúa adivinar salidas sin justificar el control y los datos.

| Entrada | Predicción | Salida observada | Coincide | Explicación |
|---:|---:|---:|---|---|
| -1 | 0 | 0 | Sí | frontera inferior: la guarda `jg` en 0x14000100d desvía a `xor eax,eax` |
| 0 | 0 | 0 | Sí | frontera del `jg`: 0 no es mayor que 0, el bucle nunca entra |
| 1 | 7 | 7 | Sí | una iteración impar: 5 + (1+1) |
| 2 | 11 | 11 | Sí | añade la rama par: 7 + (2×2) |
| 3 | 15 | 15 | Sí | 11 + (3+1), rama impar |
| 4 | 23 | 23 | Sí | 15 + (4×2), rama par |
| 5 | 29 | 29 | Sí | 23 + (5+1) |
| 6 | 41 | 41 | Sí | 29 + (6×2) |
| 10 | 95 | 95 | Sí | diez iteraciones alternando ambas ramas |
| 20 | 335 | 335 | Sí | frontera superior del rango declarado |
| 21 | 357 | 357 | Sí | fuera del rango declarado; el bucle sigue siendo contado y no desborda |

Verificación hecha ejecutando la muestra Java autorizada (`java -cp . Mystery <x>`), que es
portable; el `mystery.exe` es PE y no se ejecutó, solo se analizó estáticamente.

### Qué se recuperó y qué no

Se recuperó el **comportamiento completo**: la guarda de entrada, los límites exactos del
bucle, la condición de paridad y las dos expresiones de acumulación, verificadas en once
entradas incluidas las fronteras. Al revelar la fuente, la correspondencia es exacta.

No se recuperó nada del **texto original**: ni el nombre de la función, ni los del acumulador
y el contador, ni el formato, ni los comentarios. Tampoco la forma sintáctica elegida: los
dos `lea` podrían provenir de `i*2` o de `i+i`, y el patrón `cdq`/`and`/`xor`/`sub` podría
venir de `i % 2 == 0` o de una comparación equivalente. Equivalencia de comportamiento no
implica recuperación del fuente.

### Material recibido

Muestra de **Kenneth Santiago Rodríguez Ramírez** (análisis individual), compilada con
MSVC `cl.exe` 19.44 sobre Windows 11 x64.

| Archivo | Formato | SHA-256 |
|---|---|---|
| `mystery.exe` | PE32+ x86-64, 6 secciones, **sin tabla de símbolos** | `0903c578768f…07403c` |
| `Mystery.class` | bytecode JVM, JDK 21 | `d9d3d08a118f…b25f5d` |

Rango válido declarado: enteros pequeños. `nm mystery.exe` responde *"no symbols"*, así que
no hay nombre `mystery` que buscar: la función se localizó rastreando desde `main`.

### Localización de la función (observación)

En el desensamblado, `main` está en `0x140001080` y en `0x1400010c5` ejecuta
`call 0x140001000`, justo antes de preparar los argumentos de la impresión. Por lo tanto la
función objetivo es la que arranca en **`0x140001000`**, al inicio de `.text`, lo cual es
coherente con MSVC `/Od`, que coloca el código del usuario primero.

### Desensamblado anotado (observaciones)

| Dirección | Instrucción | Lectura |
|---|---|---|
| `140001000` | `mov [rsp+0x8],ecx` | guarda el argumento recibido en `ecx` |
| `140001008` | `cmp [rsp+0x20],0x0` | compara el argumento contra 0 |
| `14000100d` | `jg 0x140001013` | **salto 1**: si es mayor que 0, entra al cuerpo |
| `14000100f` | `xor eax,eax` → `jmp 0x140001069` | si no, devuelve 0 |
| `140001013` | `mov [rsp+0x4],0x5` | inicializa un acumulador en **5** |
| `14000101b` | `mov [rsp],0x1` | inicializa un contador en **1** |
| `140001024` | `inc eax` → `mov [rsp],eax` | incremento del contador |
| `140001030` | `cmp [rsp],eax` + `jg 0x140001065` | **salto 2**: límite del bucle, sale si contador > argumento |
| `140001038` | `cdq` / `and eax,0x1` / `xor eax,edx` / `sub eax,edx` | resto con signo entre 2 (patrón típico de `% 2`) |
| `140001042` | `jne 0x140001054` | **salto 3**: si el resto no es cero, rama impar |
| `14000104b` | `lea eax,[rax+rcx*2]` | acumulador + contador × 2 |
| `14000105b` | `lea eax,[rax+rcx*1+0x1]` | acumulador + contador + 1 |
| `140001063` | `jmp 0x140001024` | cierre del bucle |
| `140001065` | `mov eax,[rsp+0x4]` → `ret` | devuelve el acumulador |

Llamadas externas dentro de la función: **ninguna**. El único `call` del programa está en
`main` (a la rutina de conversión del argumento y a la de impresión).

### Inferencias

Separando lo medido de lo deducido: lo anterior son observaciones directas del
desensamblado. De ellas se infiere que hay **una guarda de entrada** que devuelve 0 para
valores no positivos, **un bucle contado** de 1 hasta el argumento inclusive (porque el
salto de salida es `jg`, no `jge`), y **una bifurcación por paridad** dentro del bucle con
dos acumulaciones distintas. Los dos `lea` son la pista más informativa: el compilador los
usa como calculadora, de modo que `[rax+rcx*2]` corresponde a sumar el doble del contador y
`[rax+rcx*1+0x1]` a sumar el contador más uno.

No se infiere nada sobre los nombres originales de la función, del acumulador ni del
contador: esa información no está en el binario.

In [16]:
import subprocess, hashlib, zipfile, shutil
from pathlib import Path

PRIV    = ROOT / 'reto_privado'    # NO se entrega: fuente + resultados de referencia
ENTREGA = ROOT / 'reto_entrega'    # esto SÍ se entrega
PRIV.mkdir(exist_ok=True); ENTREGA.mkdir(exist_ok=True)

MYSTERY_C = r"""#include <stdio.h>
#include <stdlib.h>

int mystery(int x) {
    int acc = 0;
    for (int i = 1; i <= x; i++) {
        if (i % 3 == 0) acc += i;
        else            acc -= 1;
    }
    return acc;
}

int main(int argc, char **argv) {
    int x = argc > 1 ? atoi(argv[1]) : 0;
    printf("%d\n", mystery(x));
    return 0;
}
"""

MYSTERY_JAVA = r"""public class Mystery {
    static int mystery(int x) {
        int acc = 0;
        for (int i = 1; i <= x; i++) {
            if (i % 3 == 0) acc += i;
            else            acc -= 1;
        }
        return acc;
    }
    public static void main(String[] args) {
        int x = args.length > 0 ? Integer.parseInt(args[0]) : 0;
        System.out.println(mystery(x));
    }
}
"""

(PRIV / 'mystery.c').write_text(MYSTERY_C, encoding='utf-8')
(PRIV / 'Mystery.java').write_text(MYSTERY_JAVA, encoding='utf-8')

# Nativo: PE32+ sin optimizar y SIN tabla de símbolos (-s), para que sea un reto real
CC  = shutil.which('gcc')
exe = ENTREGA / 'mystery.exe'
subprocess.run([CC, '-O0', '-s', str(PRIV/'mystery.c'), '-o', str(exe)], check=True)

# Java sin depuración + JAR ejecutable
JDK_BIN = Path(r"C:\Program Files\Java\jdk-17\bin")
JAVAC, JAVA, JAR = JDK_BIN/"javac.exe", JDK_BIN/"java.exe", JDK_BIN/"jar.exe"
classdir = PRIV / 'classes'; classdir.mkdir(exist_ok=True)
subprocess.run([str(JAVAC), '-g:none', '-d', str(classdir), str(PRIV/'Mystery.java')], check=True)
jar = ENTREGA / 'mystery.jar'
subprocess.run([str(JAR), '--create', '--file', str(jar), '--main-class', 'Mystery',
                '-C', str(classdir), 'Mystery.class'], check=True)

# Resultados de referencia: se quedan con nosotros para la verificación conjunta
ref = []
for x in range(0, 21):
    n = subprocess.run([str(exe), str(x)], capture_output=True, text=True).stdout.strip()
    j = subprocess.run([str(JAVA), '-jar', str(jar), str(x)], capture_output=True, text=True).stdout.strip()
    assert n == j, f"Discrepancia C/Java en x={x}: {n} vs {j}"
    ref.append((x, n))
(PRIV / 'resultados_referencia.txt').write_text(
    "x\tresultado\n" + "\n".join(f"{x}\t{n}" for x, n in ref), encoding='utf-8')

def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
h_exe, h_jar = sha(exe), sha(jar)

readme = f"""RETO DE INGENIERIA INVERSA - material para la pareja receptora
================================================================

Autores de la muestra: Juan David Cortes, Sebastian Prada, Valentina Lasprilla
IS753 Compiladores - Universidad Tecnologica de Pereira

ARCHIVOS
  mystery.exe   ejecutable nativo
  mystery.jar   version Java equivalente

PLATAFORMA Y ARQUITECTURA
  mystery.exe : PE32+ (Windows), x86-64 / AMD64
                MinGW-gcc sobre Windows 10 (19045)
                banderas: -O0 -s  (sin optimizar, sin tabla de simbolos)
  mystery.jar : bytecode JVM, major 61 (Java SE 17), portable
                javac 17.0.6, bandera -g:none

COMO SE EJECUTA
  mystery.exe 7
  java -jar mystery.jar 7
  Recibe un entero por argumento e imprime el resultado. Sin argumento asume 0.

RANGO VALIDO DE ENTRADAS
  0 a 20 (enteros). Los resultados son pequenos: no hay desbordamiento y las
  versiones C y Java coinciden en todo el rango.

FIRMAS SHA-256
  mystery.exe  {h_exe}
  mystery.jar  {h_jar}

NOTA
  La funcion es int mystery(int x), determinista, con una condicion y un bucle
  acotado. El codigo fuente NO se incluye: se revela al final del ejercicio,
  segun el enunciado. Conservamos fuente y resultados de referencia.
"""
(ENTREGA / 'README.txt').write_text(readme, encoding='utf-8')

zip_path = ROOT / 'reto_mystery_CortesPradaLasprilla.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ('mystery.exe', 'mystery.jar', 'README.txt'):
        z.write(ENTREGA / f, f)

print("ZIP listo:", zip_path)
print("Contenido:", zipfile.ZipFile(zip_path).namelist())
print("\nSHA-256 mystery.exe:", h_exe)
print("SHA-256 mystery.jar:", h_jar)
print("\nResultados de referencia (quedan en reto_privado, NO se entregan):")
print("  " + ", ".join(f"{x}->{n}" for x, n in ref))

ZIP listo: C:\Users\Tina_\executable_lab\reto_mystery_CortesPradaLasprilla.zip
Contenido: ['mystery.exe', 'mystery.jar', 'README.txt']

SHA-256 mystery.exe: d1bdfae4a9b3f7a4e30bf6087589b231b92e0b1c6c94642ed5b30646b6141730
SHA-256 mystery.jar: c815b4c9d7507671e8b7cc8ae099ceb17c5d758de1e9ac6bad6591b0c16f92ac

Resultados de referencia (quedan en reto_privado, NO se entregan):
  0->0, 1->-1, 2->-2, 3->1, 4->0, 5->-1, 6->5, 7->4, 8->3, 9->12, 10->11, 11->10, 12->22, 13->21, 14->20, 15->35, 16->34, 17->33, 18->51, 19->50, 20->49


In [17]:
def reconstructed(x):
    """Reconstruccion de mystery(x) a partir del desensamblado de mystery.exe.
    Guarda de entrada en 0x140001008, bucle contado 0x140001024-0x140001063,
    bifurcacion por paridad en 0x140001042."""
    if x <= 0:
        return 0
    acc = 5                                   # mov [rsp+0x4],0x5
    for i in range(1, x + 1):                 # i=1 .. x  (salida con jg, inclusive)
        if i % 2 == 0:
            acc += i * 2                      # lea eax,[rax+rcx*2]
        else:
            acc += i + 1                      # lea eax,[rax+rcx*1+0x1]
    return acc

# Salidas observadas ejecutando la muestra autorizada: java -cp . Mystery <x>
observations = {-1: 0, 0: 0, 1: 7, 2: 11, 3: 15, 4: 23,
                5: 29, 6: 41, 10: 95, 20: 335, 21: 357}

assert len(observations) >= 8
for x, expected in observations.items():
    assert reconstructed(x) == expected, (x, expected, reconstructed(x))
print(f"{len(observations)} entradas verificadas, todas coinciden.")

11 entradas verificadas, todas coinciden.


## 9. Ampliación opcional: análisis dinámico
Sobre su propia muestra, use GDB, LLDB o un depurador Windows. Deténgase en la función de interés; capture argumentos, registros, dirección de una instrucción y el valor de retorno. En Java puede usar jdb o el depurador del IDE y contrastar con javap.

Preguntas: ¿qué información aporta ejecutar que no provee la cabecera? ¿Se conservan las direcciones entre ejecuciones? ¿Qué limita la depuración al usar O2? No interprete la ausencia de una función independiente como prueba de que desapareció su comportamiento: pudo integrarse mediante inlining.

No se exige parchear ejecutables. Si hace un experimento adicional, modifique y recompile la fuente propia, registre hashes y compare bytes; en macOS alterar bytes puede invalidar firmas.

## 10. Informe comparativo y preguntas de sustentación
Complete con mediciones de su pareja y datos identificados del intercambio:

| Pregunta | ELF64 | PE32+ | Mach-O 64 | Java Class / JAR |
|---|---|---|---|---|
| Muestra, autor y SHA-256 | `sample_O0` (ELF64), 17.944 B, misma fuente `sample.c` compilada y medida en entorno Linux x86-64 con gcc 13.3.0 durante la preparación del taller, no en la estación Windows | `sample_O0.exe`, 255.409 B, autoría propia, SHA-256 `83d357c6af…39705d`; `sample_O2.exe` 254.296 B, `9519e17261…2609a6` | *no obtenido; ver nota al final de esta sección* | `Sample.class`, 1.307 B, `67d288243d…4431bc`; sin depuración 1.054 B, `8a2e6a2f09…bdff54`; `sample.jar` 1.216 B, `4d2b8aeb75…73c207` |
| Arquitectura o versión | ELF64, little-endian, Machine `Advanced Micro Devices X86-64`, tipo `DYN` (ejecutable independiente de posición, PIE) | PE32+ (64 bits), Machine `0x8664` = x86-64/AMD64, Windows 10 19045 | | Sin arquitectura de CPU. minor 0, major **61** = Java SE 17 |
| Firma y bytes | `7f 45 4c 46` (`\x7fELF`) en 0x00; `EI_CLASS=2` (64 bits), `EI_DATA=1` (little-endian) | `4d 5a` (`MZ`) en 0x00; `PE\0\0` en 0x80, localizado por el puntero de 0x3c. Little-endian | | `ca fe ba be` en 0x00. **Big-endian** |
| Ubicación del código | Sección `.text`: offset `0x1080`, VA `0x1080`, tamaño `0x1a1`. `transform` en `0x1169` | Sección `.text`: offset 0x600, RVA 0x1000, VMA 0x140001000, tamaño 0x7148. `transform` en VA 0x1400014a4 = offset 0xaa4 | | Atributo `Code` de cada método dentro de `methods[]`. No hay secciones ni direcciones; `transform` ocupa los offsets 0–27 de su propio Code |
| Inicio y carga | `e_entry = 0x1080`. La carga la describen 13 Program Headers, con 4 segmentos `PT_LOAD`; además 37 secciones | `AddressOfEntryPoint` (campo en 0xa8) = RVA 0x13f0 → VA 0x1400013f0 = `mainCRTStartup`, **no** `main`. Section Table en 0x188, 20 secciones | | `main([Ljava/lang/String;)V`; `Main-Class: Sample` en `META-INF/MANIFEST.MF`. El cargador ejecuta `<clinit>` antes del primer uso |
| Símbolos/metadatos recuperables | Tabla de símbolos no despojada (`nm`: `transform` 0x1169, `main` 0x11ae, `banner` 0x2010, `bias` 0x4010, `workspace` 0x4040) más `.debug_*` DWARF por `-g` | Tabla de símbolos COFF incrustada por MinGW (`nm`: `transform`, `main`, `bias`, `banner`, `workspace`) + DWARF en `.debug_*` por `-g`. Con MSVC dependerían de un PDB externo | | `constant_pool`: 67 entradas con `-g`, 58 con `-g:none`. Los nombres de métodos y descriptores **sobreviven** a `-g:none` porque son estructurales |
| Dependencias | `libc.so.6` (entrada `NEEDED` de la sección dinámica); intérprete `/lib64/ld-linux-x86-64.so.2` | Import Table: `KERNEL32.dll` y `msvcrt.dll`. Llamada concreta: `call <printf>` dentro de `main` | | Resueltas por el pool en ejecución: `Integer.parseInt`, `System.out`, `PrintStream.println`, `StringConcatFactory` vía `invokedynamic` |
| Herramienta y evidencia | `readelf -h/-l/-S/-d`, `nm`, `file` | `objdump -f/-h/-x/-d`, `nm`, y parser propio en Python sobre los bytes. Archivos `native_evidence_*.txt` | | `javap -c -p -s -v`, módulo `zipfile`, y parser propio. Archivo `java_evidence.txt` |

**El BSS en los tres formatos.** El cuarto segmento `PT_LOAD` del ELF declara
`FileSiz 0x264` contra `MemSiz 0x1290`, y su `.bss` es de tipo `NOBITS`. Es el mismo
fenómeno que en PE, donde `.bss` tiene offset de archivo 0 y carece de la bandera
`CONTENTS`, pero expresado con otra notación: ELF lo dice en el segmento, PE en las
banderas de la sección. En Java no existe el equivalente: `bias` no es un dato del archivo
sino el código `bipush 7` / `putstatic` dentro de `<clinit>`.

Responda con referencia a sus muestras:
1. ¿Por qué no basta con conocer el sistema operativo para ejecutar un binario?
2. ¿En qué se diferencian ensamblar, desensamblar y descompilar?
3. ¿Qué parte del trabajo del enlazador se puede reconocer en el archivo final?
4. ¿Qué tienen en común una tabla de símbolos nativa y el constant pool? ¿En qué difieren?
5. ¿Por qué javap puede mostrar nombres de métodos aunque compile con -g:none?
6. ¿Puede el código de máquina recuperar exactamente un AST original? Justifique con optimizaciones.
7. ¿Cómo se relaciona este análisis con el backend de un compilador y con una VM de bytecode?
8. Proponga una afirmación inicial que sus mediciones hayan refutado.

**1. ¿Por qué no basta conocer el sistema operativo para ejecutar un binario?**
Porque hacen falta además la arquitectura de CPU, el ABI y las dependencias.
`sample_O0.exe` es PE (formato de Windows) pero declara Machine `0x8664`: en un Windows
sobre ARM64 no se ejecuta de forma nativa, pese a ser el mismo SO y el mismo formato.
Y aunque coincidan formato y CPU, su Import Table exige `KERNEL32.dll` y `msvcrt.dll`
presentes en el sistema. Formato, CPU, ABI y dependencias son cuatro condiciones
independientes.

**2. ¿En qué se diferencian ensamblar, desensamblar y descompilar?**
Ensamblar traduce texto en ensamblador a bytes de máquina; es determinista y casi
biyectivo. Desensamblar es el inverso, de bytes a ensamblador: es lo que hicimos con
`objdump -d`, fiel instrucción por instrucción, pero sin recuperar nombres ni estructuras
de alto nivel. Descompilar intenta llegar a un lenguaje de alto nivel reconstruyendo
condicionales, bucles y expresiones; es heurístico y no recupera el texto original.
Nosotros desensamblamos: lo que leímos fue `push %rbp`, nunca código C.

**3. ¿Qué parte del trabajo del enlazador se puede reconocer en el archivo final?**
Cuatro rastros concretos en nuestra muestra. La tabla de 20 secciones, resultado de fusionar
los objetos de entrada en `.text`, `.data`, `.rdata` y `.bss`. Las direcciones ya resueltas:
`transform` quedó fijado en 0x1400014a4, decisión del enlazado, no del compilador. La
`ImageBase` 0x140000000 y el punto de entrada apuntando a `mainCRTStartup`, runtime que el
enlazador agregó. Y la Import Table, que es precisamente lo que el enlazador **no** resolvió
y delegó al cargador.

**4. ¿Qué tienen en común una tabla de símbolos nativa y el constant pool? ¿En qué difieren?**
Ambos asocian nombres con entidades del programa y permiten que las herramientas muestren
identificadores legibles. Difieren en tres cosas. Primero, en obligatoriedad: la tabla de
símbolos del PE es prescindible para ejecutar (el cargador no la consulta y puede eliminarse),
mientras el constant pool es indispensable porque la JVM resuelve cada llamada y cada campo
a través de él en ejecución. Segundo, en estructura: el pool es tipado y referencial
(`Methodref` → `Class` + `NameAndType` → `Utf8`), sin duplicar ningún texto; la tabla nativa
es plana, nombre contra dirección. Tercero, en alcance: el pool también guarda las constantes
del programa, como la cadena `COMPILADORES_LAB_2026`, mientras en el PE esa cadena vive en
una sección aparte, `.rdata`.

**5. ¿Por qué `javap` puede mostrar nombres de métodos aunque compile con `-g:none`?**
Porque esos nombres no son información de depuración. Los nombres de métodos, campos y sus
descriptores están en el constant pool y son estructuralmente necesarios: sin ellos la JVM
no puede resolver `invokestatic #25 // Method transform:(I)I`. `-g:none` solo elimina
`LineNumberTable`, `LocalVariableTable` y `SourceFile`. Nuestra medición lo confirma: en
`java_no_debug` siguen apareciendo `transform`, `main` y `bias`; lo que desapareció fueron
los nombres de las variables locales (`x`, `args`, `this`) y el del archivo fuente.

**6. ¿Puede el código de máquina recuperar exactamente un AST original? Justifique.**
No. La traducción no es invertible y nuestra comparación O0/O2 lo demuestra. En O2, el
`x * 3` del fuente se convirtió en un único `lea (%rcx,%rcx,2),%eax`: el nodo de
multiplicación del AST ya no existe, quedó expresado como un cálculo de dirección.
Desapareció el prólogo de la función, y el único punto de salida del fuente se volvió tres
`ret` distintos. Además, muchos árboles sintácticos distintos compilan al mismo código
máquina, así que ni siquiera hay una preimagen única. Y lo que nunca estuvo en el binario
—comentarios, nombres de locales, formato, la forma elegida de escribir el control de
flujo— no puede recuperarse de ningún modo. Se recupera comportamiento equivalente, no texto.

**7. ¿Cómo se relaciona este análisis con el backend de un compilador y con una VM de bytecode?**
El backend de un compilador termina produciendo exactamente los archivos que analizamos:
selecciona instrucciones, asigna registros y luego escribe secciones, símbolos y
reubicaciones en el formato del sistema. La diferencia visible entre nuestro O0 y nuestro O2
—`x` en pila contra `x` en `%ecx`— es la etapa de optimización y asignación de registros
hecha observable en la salida. En Java ese backend se parte en dos: `javac` llega solo hasta
el bytecode de una máquina de pila (`iload_0`, `imul`, `ireturn`), dejando sin decidir los
registros, y el JIT de la JVM completa la selección de instrucciones y la asignación de
registros en tiempo de ejecución. Eso explica por qué el bytecode es portable y el PE no.

**8. Afirmación inicial que nuestras mediciones refutaron.**
Dos. La primera: suponíamos que compilar con `-O2` haría desaparecer `transform` por
inlining. Sigue presente como función independiente en ambos binarios, porque `bias` está
declarado `volatile` y eso impide al optimizador fusionar la función. La segunda, y más
instructiva: suponíamos que las métricas globales del binario (tamaño, número de `call`,
número de saltos) mostrarían el efecto del optimizador. No lo muestran: 333 llamadas en
ambos, 1.255 contra 1.252 saltos, 1.113 bytes de diferencia sobre 255.000. El runtime de
MinGW domina el binario y diluye por completo nuestro código, así que hubo que aislar la
función para ver cualquier diferencia.

## 11. Entrega y rúbrica
Entregue un ZIP con el cuaderno diligenciado, carpeta de fuentes propias (reveladas al final), binarios, salidas de herramientas y README con pasos de reproducción. Indique qué celdas no pudo ejecutar y por qué. Evite adjuntar carpetas completas de SDK o del JDK.

| Criterio | Peso | Evidencia para desempeño completo |
|---|---:|---|
| Investigación y comparación | 15% | Cuatro formatos, fuentes primarias y distinciones correctas |
| Cabeceras y mapa nativo | 25% | Campos validados, mapa medido y conversión de dirección comprobada |
| Java Class/JAR | 20% | Pool, Code, pila, manifiesto y compatibilidad explicados |
| Reconstrucción y pruebas | 25% | Hipótesis desde instrucciones, flujo y ocho casos de prueba |
| Reproducibilidad y sustentación | 15% | Versiones, hashes, comandos y explicación individual |

Para cada criterio asigne nivel 0 a 5: 0 sin evidencia; 1 errores centrales; 2 parcial; 3 correcto con omisiones; 4 completo y reproducible; 5 completo, validado y con discusión de límites. **Nota final = suma(peso × nivel)** usando pesos decimales; escala 0–5.

## 12. Fuentes primarias y lecturas
- [Apple: definiciones Mach-O en loader.h](https://github.com/apple-oss-distributions/xnu/blob/main/EXTERNAL_HEADERS/mach-o/loader.h). Consultar mach_header_64, segment_command_64 y entry_point_command.
- [Microsoft: PE/COFF Format](https://learn.microsoft.com/en-us/windows/win32/debug/pe-format). Consultar cabeceras, RVA, Section Table e importaciones.
- [Linux man-pages: elf(5)](https://man7.org/linux/man-pages/man5/elf.5.html). Consultar ELF header, Program header y Section header.
- [Oracle JVMS: Class File Format](https://docs.oracle.com/javase/specs/jvms/se25/html/jvms-4.html). Consultar ClassFile, pool y Code.
- [Oracle: instrucciones JVM](https://docs.oracle.com/javase/specs/jvms/se25/html/jvms-6.html). Consultar las instrucciones observadas.
- [Oracle: javap](https://docs.oracle.com/en/java/javase/25/docs/specs/man/javap.html) y [especificación JAR](https://docs.oracle.com/en/java/javase/25/docs/specs/jar/jar.html).
- [GNU Binutils](https://sourceware.org/binutils/docs/binutils/) y [Ghidra, repositorio oficial](https://github.com/NationalSecurityAgency/ghidra).

Las referencias Java SE 25 sirven de documentación; las actividades básicas también se plantean para JDK 17+. Anote la versión realmente utilizada y compruebe diferencias de opciones. Las salidas y distribuciones concretas dependen del compilador y del enlazador.

In [18]:
# ===== AUDITORIA FINAL: compare con lo escrito en las tablas =====
import hashlib, struct, zipfile
from pathlib import Path
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()

print("### HASHES (matriz comparativa + README) ###")
for f in ['sample_O0.exe','sample_O2.exe','Sample.class','sample.jar']:
    p = ROOT/f
    if p.exists(): print(f"  {f:22} {p.stat().st_size:>8} B  {sha(p)}")
nd = ROOT/'java_no_debug'/'Sample.class'
if nd.exists(): print(f"  {'Sample.class (-g:none)':22} {nd.stat().st_size:>8} B  {sha(nd)}")

print("\n### PE: cabecera (tabla 'Por completar') ###")
b = (ROOT/'sample_O0.exe').read_bytes()
pe  = struct.unpack_from('<I', b, 0x3c)[0]
opt = pe + 24
rva = struct.unpack_from('<I', b, opt+16)[0]
ib  = struct.unpack_from('<Q', b, opt+24)[0]
print(f"  firma 0x00          : {b[0:2].hex(' ')}")
print(f"  PE offset (de 0x3c) : {hex(pe)}  -> {b[pe:pe+4]}")
print(f"  Machine en {hex(pe+4):<6}   : {b[pe+4:pe+6].hex(' ')} -> {hex(struct.unpack_from('<H',b,pe+4)[0])}")
print(f"  Secciones           : {struct.unpack_from('<H',b,pe+6)[0]}")
print(f"  EntryPoint en {hex(opt+16):<5} : {b[opt+16:opt+20].hex(' ')} -> RVA {hex(rva)}")
print(f"  ImageBase           : {hex(ib)}  -> VA entry {hex(ib+rva)}")
print(f"  Section Table en    : {hex(opt + struct.unpack_from('<H',b,pe+20)[0])}")

print("\n### Java (Actividad E) ###")
cb = (ROOT/'Sample.class').read_bytes()
minor, major, pool = struct.unpack_from('>HHH', cb, 4)
print(f"  magic               : {cb[:4].hex(' ')}")
print(f"  minor / major       : {minor} / {major}   (major {major} = Java SE {major-44})")
print(f"  constant_pool_count : {pool}   (entradas reales: {pool-1})")

print("\n### Actividad D: conversion de direccion ###")
off = 0x600 + (0x1400014a4 - 0x140001000)
got = b[off:off+8].hex(' ')
print(f"  offset calculado    : {hex(off)}")
print(f"  bytes ahi           : {got}")
print(f"  {'OK: coincide con el inicio de transform' if got.startswith('55 48 89 e5') else 'OJO: las direcciones se movieron, rehaz la Actividad D'}")

print("\n### ZIP del reto ###")
z = ROOT/'reto_mystery_CortesPradaLasprilla.zip'
print(f"  existe: {z.exists()}", zipfile.ZipFile(z).namelist() if z.exists() else "")

### HASHES (matriz comparativa + README) ###
  sample_O0.exe            255409 B  68774e2b0ab21025732b26d53a69990668ca7d3b034c36670d5ef90318fe5561
  sample_O2.exe            254296 B  45ef1ab5bbab019f487fa261d986048002f9eee6ca65381395f10764ef7b24a6
  Sample.class               1307 B  67d288243daa39d60aa28fdecf171b5b57f5999e22191b6a2e11dd78934431bc
  sample.jar                 1216 B  c3ca3f7721bd35531cadd53b645c704873d7aa61e0c5d47a1e47b0417deddb39
  Sample.class (-g:none)     1054 B  8a2e6a2f0982b0a786281f4f24a6f82612e60f89df10ee82fcc4f4c052bdff4e

### PE: cabecera (tabla 'Por completar') ###
  firma 0x00          : 4d 5a
  PE offset (de 0x3c) : 0x80  -> b'PE\x00\x00'
  Machine en 0x84     : 64 86 -> 0x8664
  Secciones           : 20
  EntryPoint en 0xa8  : f0 13 00 00 -> RVA 0x13f0
  ImageBase           : 0x140000000  -> VA entry 0x1400013f0
  Section Table en    : 0x188

### Java (Actividad E) ###
  magic               : ca fe ba be
  minor / major       : 0 / 61   (major 61 = Java 

**Nota de reproducibilidad.** Al recompilar la misma fuente con las mismas banderas, los
binarios nativos cambian de SHA-256 (`sample_O0.exe` pasó de `d0265afa…` a `18677f24…` y
luego a `83d357c6…`) manteniendo el tamaño exacto de 255.409 bytes: el enlazador de MinGW
incrusta metadatos de compilación. En cambio `Sample.class` conserva su hash
`67d288243d…4431bc` entre compilaciones, porque `javac` es determinista. Conclusión: el
hash identifica un artefacto concreto, no una fuente; y la reproducibilidad byte a byte
depende de la cadena de herramientas, no del lenguaje.